In [2]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [3]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')

In [4]:
start = dt.datetime(2019,6,18)
start1 = str(start)
end = dt.datetime(2019,6,26)
end1 = str(end)
print(start,end)

2019-06-18 00:00:00 2019-06-26 00:00:00


In [5]:
app_version = '2.0.2'

# new user reference

In [6]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (
    f"""SELECT
    * FROM 
    `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
)
new_user_reference = client.query(query).to_dataframe()

In [7]:
new_user_reference['user_first_touch_timestamp'] = new_user_reference['user_first_touch_timestamp'].astype('datetime64[s]')
android_new = new_user_reference[(new_user_reference['user_first_touch_timestamp'] >= start)]

# Ftue completed users

In [8]:
query = (f"""
             SELECT 
             user_id as device_id,
             event_timestamp
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start1.replace('-','')}"
             AND "{end1.replace('-','')}"
             AND app_info.version = '{app_version}'
            AND device.operating_system = 'ANDROID'
             AND params.key = 'action'
            AND event_name IN ('natasha')
           AND params.value.string_value = 'hand_pointer_achievements_clicked'""")
         
ftue_complete = client.query(query).to_dataframe() 

In [9]:
ftue_complete.columns = ['device_id','create_time']
ftue_complete.sort_values('create_time',inplace=True,ascending=False)
ftue_complete.drop_duplicates('device_id',inplace=True)
ftue_complete = ftue_complete[ftue_complete['device_id'].isin(android_new['device_id'])]
ftue_complete['create_time'] = pd.to_datetime(ftue_complete['create_time'], unit = 'us')
print(len(ftue_complete))
ftue_complete.head()

675


,device_id,create_time
502,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007
485,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007
516,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008
469,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008
496,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007


In [10]:
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$gte': start}},{"sign_up_details",'login_details.last_request_at'}): # end condition
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
users = pd.DataFrame(dic_flattened)
users = users[["_id","sign_up_details_device_id",'login_details_last_request_at']]
users.columns = ["user_id","device_id",'last_request']
len(users)

2792

In [11]:
users.sort_values(['device_id','last_request'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
print(len(users))

2717


In [12]:
ftue_complete_user = pd.merge(ftue_complete,users,on='device_id')
ftue_complete_user = ftue_complete_user[['user_id','device_id','create_time']]

In [13]:
print(len(ftue_complete_user))
ftue_complete_user.head()

675


,user_id,device_id,create_time
0,5d13b4587119e60013b3e9d6,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007
1,5d1249a7c5ebf8001185851f,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007
2,5d13a7f8114e23001a9596e9,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008
3,5d13a4b4114e23001a954a35,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008
4,5d1112c4fa80e30028d2b431,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007


# number of users who have trained

In [14]:
query = (f"""
             SELECT user_id as device_id,
                    event_timestamp
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start1.replace('-','')}"
             AND "{end1.replace('-','')}"
            AND params.key = 'action'
             AND app_info.version = '{app_version}'
          AND params.value.string_value IN ('sent_to_training','sent_to_training_using_hitcoins')
           
         """)
df = client.query(query).to_dataframe()

In [15]:
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'],unit='us')

In [16]:
training_users = pd.merge(ftue_complete,df,on='device_id')
training_users = training_users[training_users['event_timestamp']>training_users['create_time']]
training_users = training_users[(training_users['event_timestamp']-training_users['create_time'])<'24:00:00']
training_users.drop_duplicates('device_id',inplace=True)

In [17]:
training_users = pd.merge(training_users,users,on='device_id')
training_users['d1'] =(training_users['last_request']-training_users['create_time']) > '24:00:00'

In [27]:
len(training_users)

335

# number of users who have used instant coins

In [18]:
c_instant_coins = cursor.superstars.user_collectables_logs
aw_instant_coins = []
for documents in c_instant_coins.aggregate([{'$unwind':"$data"}, 
                    {"$match" : {"data.reason_type" : 'QUICK_TRAINING',  # the quick training is actually instant training in the game
                                  "type":"HARD_CURRENCY",                # the 'INSTANT' word appers next to "TRAIN"
                                  'data.quantity': {'$lt': 0},           # here this is saved as quick training
                                  'data.created_at': {'$gte': start}}}]):
    aw_instant_coins.append(documents)
    
dic_flattened = [flatten(d) for d in aw_instant_coins]
instant_coins = pd.DataFrame(dic_flattened)
instant_coins = instant_coins[instant_coins["user"].isin(training_users['user_id'])]
instant_coins = instant_coins[["_id","data_created_at",'user']]
instant_coins.columns = ["instant_coin_id", "instant_coin_used_at","user_id"]

In [19]:
print(len(instant_coins))
instant_coins.head()

863


,instant_coin_id,instant_coin_used_at,user_id
1850,5d0d61cdfa80e30028275159,2019-06-21 23:01:33.249,5d0d60338185c2001923106d
1852,5d0d85b7fa80e300282d9e67,2019-06-22 12:29:46.757,5d0d8421fa80e300282d5aec
1853,5d0d85b7fa80e300282d9e67,2019-06-22 12:30:03.871,5d0d8421fa80e300282d5aec
1856,5d0d98358185c200192d0361,2019-06-22 02:53:41.795,5d0d96be8185c200192ce955
1857,5d0d98358185c200192d0361,2019-06-22 02:58:35.728,5d0d96be8185c200192ce955


In [20]:
instant_users = pd.merge(ftue_complete_user,instant_coins,on='user_id')
instant_users = instant_users[instant_users['instant_coin_used_at']-instant_users['create_time']<'24:00:00']
len(instant_users)

769

In [21]:
grouped = instant_users.groupby('user_id').agg({'instant_coin_id':'count'}).reset_index()
print(len(grouped))
grouped.head()

177


,user_id,instant_coin_id
0,5d0d60338185c2001923106d,1
1,5d0d8421fa80e300282d5aec,2
2,5d0d8c7b8185c200192aa41f,1
3,5d0d9197fa80e30028311af8,1
4,5d0d96be8185c200192ce955,16


# planning and grouping

In [22]:
grouped_zero = pd.merge(grouped,training_users[['user_id','d1']],on='user_id',how='right')
grouped_zero = grouped_zero.fillna(0)
len(grouped_zero)

335

In [23]:
grouped_zero['instant_coin_id'] = grouped_zero['instant_coin_id'].astype(int)
grouped_zero.head()

,user_id,instant_coin_id,d1
0,5d0d60338185c2001923106d,1,False
1,5d0d8421fa80e300282d5aec,2,False
2,5d0d8c7b8185c200192aa41f,1,False
3,5d0d9197fa80e30028311af8,1,False
4,5d0d96be8185c200192ce955,16,True


In [24]:
distribution = grouped_zero.groupby('instant_coin_id').agg({'user_id':'count','d1':'sum'})

In [25]:
distribution['user_cumsum'] = distribution.loc[::-1,'user_id'].cumsum()[::-1]
distribution['d1_cumsum'] = distribution.loc[::-1,'d1'].cumsum()[::-1]

In [26]:
distribution

,user_id,d1,user_cumsum,d1_cumsum
instant_coin_id,,,,
0,158,44.0,335,92.0
1,65,14.0,177,48.0
2,27,8.0,112,34.0
3,16,2.0,85,26.0
4,18,5.0,69,24.0
5,6,2.0,51,19.0
6,6,2.0,45,17.0
7,3,2.0,39,15.0
8,5,0.0,36,13.0
